In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"
scenario = "intervention"

In [3]:
# Parameters
location = "india"
vehicle = "rice"
scenario = "intervention"

In [4]:
def aggregate_by_scenario(df):
    return (
        df.groupby(["scenario", "input_draw", "wealth_quintile"])
        .value.sum()
        .groupby(["scenario", "wealth_quintile"])
        .mean()
    )

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = (
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        )
        .assign(value=0)
        .assign(scenario=lambda x: x.scenario.replace("intervention", scenario))
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,lowest,baseline,50,0,5.798647
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,second,baseline,50,0,0.000000
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,middle,baseline,50,0,0.000000
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,fourth,baseline,50,0,5.649963
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,highest,baseline,50,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
719995,person_time,impairment,anemia,severe,95_plus,severe,lowest,intervention,144,0,0.000000
719996,person_time,impairment,anemia,severe,95_plus,severe,second,intervention,144,0,0.000000
719997,person_time,impairment,anemia,severe,95_plus,severe,middle,intervention,144,0,0.000000
719998,person_time,impairment,anemia,severe,95_plus,severe,fourth,intervention,144,0,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

mild          180000
moderate      180000
not_anemic    180000
severe        180000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      fourth             4.108142e+06
              highest            3.972768e+06
              lowest             5.750990e+06
              middle             4.378575e+06
              second             4.848110e+06
intervention  fourth             4.108146e+06
              highest            3.972775e+06
              lowest             5.751000e+06
              middle             4.378577e+06
              second             4.848114e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      fourth             1.948916e+06
              highest            1.572980e+06
              lowest             3.302735e+06
              middle             2.256069e+06
              second             2.647889e+06
intervention  fourth             1.908500e+06
              highest            1.530661e+06
              lowest             3.251139e+06
              middle             2.216690e+06
              second             2.607985e+06
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      fourth             0.474403
              highest            0.395941
              lowest             0.574290
              middle             0.515252
              second             0.546169
intervention  fourth             0.464565
              highest            0.385288
              lowest             0.565317
              middle             0.506258
              second             0.537938
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/{scenario}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,lowest,49649.700497
1,Female,0.0,0.019178,not_pregnant,second,43575.622144
2,Female,0.0,0.019178,not_pregnant,middle,38689.356750
3,Female,0.0,0.019178,not_pregnant,fourth,36440.833935
4,Female,0.0,0.019178,not_pregnant,highest,29202.585338
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,lowest,15176.692284
281,Male,95.0,125.000000,not_pregnant,second,15848.509818
282,Male,95.0,125.000000,not_pregnant,middle,16370.176208
283,Male,95.0,125.000000,not_pregnant,fourth,17265.313670


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
fourth     3.799765e+06
highest    3.668830e+06
lowest     5.312734e+06
middle     4.040741e+06
second     4.479329e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      fourth             1.802621e+06
              highest            1.452639e+06
              lowest             3.051049e+06
              middle             2.081999e+06
              second             2.446472e+06
intervention  fourth             1.765237e+06
              highest            1.413555e+06
              lowest             3.003380e+06
              middle             2.045658e+06
              second             2.409601e+06
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = (
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        )
        .assign(value=0)
        .assign(scenario=lambda x: x.scenario.replace("intervention", scenario))
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,lowest,baseline,50,0,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,second,baseline,50,0,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,middle,baseline,50,0,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,fourth,baseline,50,0,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,highest,baseline,50,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
359995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,lowest,intervention,144,0,0.0
359996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,second,intervention,144,0,0.0
359997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,middle,intervention,144,0,0.0
359998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,fourth,intervention,144,0,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['maternal_disorders_to_recovered_from_maternal_disorders',
       'no_transition',
       'susceptible_to_maternal_disorders_to_maternal_disorders'],
      dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      fourth             4.144405e+06
              highest            3.748285e+06
              lowest             6.062086e+06
              middle             4.604932e+06
              second             5.129029e+06
intervention  fourth             4.125592e+06
              highest            3.728929e+06
              lowest             6.036492e+06
              middle             4.585940e+06
              second             5.111209e+06
Name: value, dtype: float64

In [18]:
path = f"./results/{location}/{vehicle}/{scenario}/maternal_disorders_incident_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"
if pathlib.Path(path).is_file():
    neonatal_deaths = pd.read_parquet(path)
else:
    neonatal_deaths = (
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .assign(
            maternal_scenario=lambda x: x.maternal_scenario.replace(
                "intervention", scenario
            )
        )
    )

neonatal_deaths = neonatal_deaths.rename(columns={"maternal_scenario": "scenario"})
neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,deaths,cause,stillborn,stillborn,0_to_6_months,Female,lowest,baseline,baseline,69,0,0.000000
1,deaths,cause,stillborn,stillborn,0_to_6_months,Female,second,baseline,baseline,69,0,0.000000
2,deaths,cause,stillborn,stillborn,0_to_6_months,Female,middle,baseline,baseline,69,0,0.000000
3,deaths,cause,stillborn,stillborn,0_to_6_months,Female,fourth,baseline,baseline,69,0,0.000000
4,deaths,cause,stillborn,stillborn,0_to_6_months,Female,highest,baseline,baseline,69,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
31995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,lowest,baseline,baseline,96,0,69.798572
31996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,second,baseline,baseline,96,0,38.776985
31997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,middle,baseline,baseline,96,0,23.266191
31998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,fourth,baseline,baseline,96,0,38.776985


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      fourth             143816.080207
              highest            139178.352858
              lowest             204967.384804
              middle             152827.851411
              second             174488.674967
intervention  fourth             143629.950682
              highest            138922.424760
              lowest             204781.255278
              middle             152711.520458
              second             174294.790044
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/{scenario}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/{scenario}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = (
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/intervention/anemia_cases.parquet"
        )
        .assign(value=0)
        .assign(scenario=lambda x: x.scenario.replace("intervention", scenario))
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,fourth,33424.279506,baseline
1,Female,0.0,0.019178,highest,26298.067393,baseline
2,Female,0.0,0.019178,lowest,47936.142149,baseline
3,Female,0.0,0.019178,middle,36378.268813,baseline
4,Female,0.0,0.019178,second,41281.501854,baseline
...,...,...,...,...,...,...
495,Male,95.0,125.000000,fourth,13068.179942,intervention
496,Male,95.0,125.000000,highest,16091.818802,intervention
497,Male,95.0,125.000000,lowest,11853.135866,intervention
498,Male,95.0,125.000000,middle,12450.025500,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      fourth             1.099594e+08
              highest            1.033711e+08
              lowest             1.274585e+08
              middle             1.161895e+08
              second             1.184645e+08
intervention  fourth             1.065478e+08
              highest            9.880798e+07
              lowest             1.246339e+08
              middle             1.130496e+08
              second             1.156244e+08
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      fourth             1.117620e+08
              highest            1.048237e+08
              lowest             1.305095e+08
              middle             1.182715e+08
              second             1.209110e+08
intervention  fourth             1.083130e+08
              highest            1.002215e+08
              lowest             1.276373e+08
              middle             1.150952e+08
              second             1.180340e+08
Name: value, dtype: float64

In [25]:
path = (
    f"./results/{location}/{vehicle}/{scenario}/prevalent_anemia_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/{scenario}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = (
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ntd_cases_by_scenario.csv"
        )
        .assign(value=0)
        .assign(scenario=lambda x: x.scenario.replace("intervention", scenario))
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
baseline      lowest             4099.443838
              second             3621.430801
              middle             3251.193731
              fourth             3065.715463
              highest            2610.941169
intervention  fourth             2976.921664
              highest            2530.355767
              lowest             3982.314525
              middle             3165.690407
              second             3533.814714
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/{scenario}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)